# Examine drop-out risks and Intervention factors
We will use a combination of 
1. XGBoost global feature importance: how the trees were built
2. Permutation importance: how much performance depends on a feature
3. SHAP global summary: how features contribute to predictions

and compare the results to derive insights on drop-out risks and intervention factors. We combine three complementary methods to ensure robustness and validity of our findings.

Note that none of these methods provide causal insights, but they can help *identify important factors* associated with student drop-out.




In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
import shap
import warnings
warnings.filterwarnings('ignore')
import os
from pathlib import Path
import joblib 
from sklearn.preprocessing import MinMaxScaler

print("Libraries imported successfully!")

In [ ]:
# Load the tuned model
current_user = os.getenv('USER')
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
final_model = joblib.load(PROCESSED_DIR / "xgb_final_tuned_trainval.pkl")  
tuning_outputs = joblib.load(PROCESSED_DIR / "xgb_tuning_outputs.pkl")

print(f"Model loaded successfully!")

In [ ]:
# Schema integrity checks (run once after loading)

# // Retrieve the exact feature schema used during model training to validate downstream importance and SHAP analysis
feature_cols = tuning_outputs["feature_cols"]

# Check that the model exposes a booster and supports probability prediction
# // Ensures the loaded artifact is a tree-based XGBoost model required for gain-based importance
assert hasattr(final_model, "get_booster"), "Loaded object does not appear to be an XGBoost model"

# // Confirms the model is suitable for probability-based evaluation and SHAP analysis
assert hasattr(final_model, "predict_proba"), "Loaded model does not support predict_proba"

# Verify feature count consistency between saved schema and model
booster = final_model.get_booster()
booster_features = booster.feature_names

# // Ensures feature names were preserved during training, which is required for correct importance mapping
assert booster_features is not None, "Booster feature names are missing"

# // Detects silent mismatches between training schema and model internals that would corrupt explanations
assert len(booster_features) == len(feature_cols), (
    f"Feature count mismatch: booster={len(booster_features)}, schema={len(feature_cols)}"
)

# Optional visibility check (safe to print once)
print(f"Schema check passed: {len(feature_cols)} features aligned between model and metadata")

## 1. XGBoost Global Feature Importance
Global feature importance provides an aggregated view of which features the XGBoost model relied on most during tree construction across all predictions. We use the (total) gain metric, which measures the total reduction in the model’s training loss contributed by splits on a feature, summed across all trees. This offers a structural, model-level perspective on feature usage, rather than a direct explanation of individual predictions.

Important notes:

- We use total gain since it reflects the full contribution of a feature across all trees rather than its average impact per split. Similarly, we use total cover to reflect the aggregate sample coverage of splits on a feature.

- Gain, cover, and weight values were normalized to relative (percentage) to allow for comparison across features and to mitigate scale differences. 

- The weight metric was not treated as a feature importance measure, but used as a diagnostic measure due to its biased toward high-cardinality or continuous features. 

- For transparency, unused features were retained with zero importance.

- Finally, gain-based importance reflects how the model was constructed, not how features contribute to predictions. As such, it should be interpreted as complementary to, rather than interchangeable with, SHAP-based explanations.

In [ ]:
# Extract XGBoost global feature importance
# Get importance scores for different metrics
importance_gain = final_model.get_booster().get_score(importance_type='total_gain')
importance_weight = final_model.get_booster().get_score(importance_type='weight')    
importance_cover = final_model.get_booster().get_score(importance_type='total_cover')

# Convert to DataFrame for easier analysis
def importance_to_df(importance_dict, importance_type, all_features):
    df = pd.DataFrame(list(importance_dict.items()),
                     columns=['feature', f'importance_{importance_type}'])
    df = df.set_index('feature').reindex(all_features).fillna(0.0).reset_index()     # Ensure unused features are included with 0 importance so results reflect the full model input space
    total = df[f'importance_{importance_type}'].sum()
    df[f'importance_{importance_type}_pct'] = (100 * df[f'importance_{importance_type}'] / total) if total > 0 else 0.0  # Normalize to % so importances are interpretable and comparable within a metric
    df = df.sort_values(f'importance_{importance_type}', ascending=False)
    return df

all_features = list(tuning_outputs["feature_cols"])  # Use the saved training feature schema to prevent silent feature-order/name mismatches in importance reporting

df_gain = importance_to_df(importance_gain, 'total_gain', all_features)
df_weight = importance_to_df(importance_weight, 'weight', all_features)
df_cover = importance_to_df(importance_cover, 'total_cover', all_features)

In [ ]:
# Visualize XGBoost global feature importance
plt.figure(figsize=(12, 12))

# Plot top 20 features by total gain
top_20_gain = df_gain.head(20)
plt.subplot(3, 1, 1)
plt.barh(range(len(top_20_gain)), top_20_gain['importance_total_gain_pct'])
plt.yticks(range(len(top_20_gain)), top_20_gain['feature'])
plt.xlabel('Importance (% of total gain)') # Label reflects that importance is expressed as percentage contribution to total training-loss reduction
plt.title('Top 20 Features - XGBoost Global Feature Importance (Total Gain)')
plt.gca().invert_yaxis()

# Plot top 20 features by weight (diagnostic only)
top_20_weight = df_weight.head(20)
plt.subplot(3, 1, 2)
plt.barh(range(len(top_20_weight)), top_20_weight['importance_weight_pct'])
plt.yticks(range(len(top_20_weight)), top_20_weight['feature'])
plt.xlabel('Split Frequency (% of total)')
plt.title('Top 20 Features - XGBoost Split Frequency (Weight, Diagnostic)') # Mark weight as diagnostic to avoid over-interpreting it as true importance
plt.gca().invert_yaxis()

# Plot top 20 features by total cover
top_20_cover = df_cover.head(20)
plt.subplot(3, 1, 3)
plt.barh(range(len(top_20_cover)), top_20_cover['importance_total_cover_pct'])
plt.yticks(range(len(top_20_cover)), top_20_cover['feature'])
plt.xlabel('Sample Coverage (% of total)') # Clarifies that cover reflects how many samples are affected by splits on a feature
plt.title('Top 20 Features - XGBoost Global Feature Importance (Total Cover)')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Print top 20 features for each global importance metric
print("\nTop 20 Features by Total Gain (% of total):")
display(df_gain[['feature', 'importance_total_gain_pct']].head(20))

print("\nTop 20 Features by Weight (Split Frequency %, Diagnostic):")
display(df_weight[['feature', 'importance_weight_pct']].head(20))

print("\nTop 20 Features by Total Cover (% of total):")
display(df_cover[['feature', 'importance_total_cover_pct']].head(20))

## 1.1 Results & interpretation

We extract the global feature importance from the trained XGBoost model and display the top features based on the `total gain` metric, as well as the `weight` metric. A high `total gain` indicates that the feature improves the model's predictive power when used for splits: the model becomes better at distinguishing between drop-outs and non-drop-outs. A high `weight` indicates that the feature is frequently used for splits in the decision trees. For complementary (diagnostic) information we also assess `cover`, which measures the relative coverage of samples affected by splits on a feature (i.e. a feature's influence). 

Overall (based on both gain and weight), the most important features are course result related: `Total Credits Block A`, `Average Grade A` and `Potential Credits A Missing Flag`. This suggests that students' academic performance in Block A is is the dominant driver of dropout predictions, indicating that early academic outcomes play a central role in distinguishing between drop-outs and non-drop-outs. Note that the feature `Potential Credits A Missing Flag` does not have a high weight, indicating that it might only be important for a subset of students. `Timing- and distance-related features` contribute additional structure by refining decision boundaries and segmenting students into subgroups. 

Examining the top feature by `cover`, we see a lot of degrees appearing (i.e. operating at broader splits). This tells us that the model segments students by degree, indicating that the model considers degree-level differences important when predicting drop-out risk: different degree programs have distinct dropout patterns. 

## 2. Permutation Importance

Permutation importance measures how much the model's performance (Average Precision/PR-AUC) decreases when each feature is randomly shuffled. This approach is complementary to tree-based feature importance because:

1. It's model-agnostic and based on actual prediction performance

2. It's less biased toward high-cardinality categorical features

3. It reflects the importance of a feature for predictive performance, including cases where effects arise through interactions with other features.

Note that permutation importance does not explicitly model interactions;it implicitly reflects interactions only insofar as breaking one feature disrupts joint effects.


In [ ]:
# Perform permutation importance analysis
# Load the test data first
df = pd.read_csv(PROCESSED_DIR / 'feature_selection.csv')

# Define target and feature columns
target_col = 'sdo5_degree_drop_out'
exclude_cols = ['set', target_col]

# Use the saved training feature schema to ensure permutation importance is computed on the exact feature space the model was trained on
feature_cols = tuning_outputs["feature_cols"]

# Extract test set
test_df = df[df['set'] == 'test'].copy()

# Enforce column order and feature alignment to prevent silent corruption of permutation importance scores
X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

# Perform permutation importance on test set
# Using multiple repetitions for more robust estimates
perm_importance = permutation_importance(
    final_model, 
    X_test, 
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
    scoring='average_precision'  # Using same metric as model tuning
)

# Create DataFrame with results
perm_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)


In [ ]:
# Visualize permutation importance
plt.figure(figsize=(12, 8))

# Plot top 20 features
top_20_perm = perm_df.head(20)
plt.barh(
    range(len(top_20_perm)),
    top_20_perm['importance_mean'],
    xerr=top_20_perm['importance_std']
)
plt.yticks(range(len(top_20_perm)), top_20_perm['feature'])

# Shortened label to emphasize that values represent performance drop, not absolute performance
plt.xlabel('Decrease in Average Precision (PR-AUC)')

# Slightly simplified title to keep focus on evaluation set and method
plt.title('Top 20 Features - Permutation Importance (Test Set)')

# Add reference line to distinguish features with positive vs near-zero/negative impact on performance
plt.axvline(0, linestyle='--', linewidth=1)

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Print top 20 features by permutation importance

print("\nTop 20 Features by Permutation Importance (PR-AUC decrease):")
display(
    perm_df[['feature', 'importance_mean', 'importance_std']]
    .head(20)
)


## 2.1 Results & interpretation

All Block A course result features rank among the most important variables under permutation importance, confirming the results from the XGBoost gain-based analysis. In particular, `Total Credits Block A` and `Average Grade A` cause the largest decreases in PR-AUC when permuted, indicating that early `academic performance` is central to the model’s predictive capability. In addition, features such as `has_dsa` and `gender` also contribute to predictive performance, although to a lesser extent than academic outcomes. This suggests that while dropout risk is primarily driven by early performance, structural and demographic factors provide complementary information.

## 3. SHAP Global Summary

The goal is to create a global summary showing the average impact of each feature. SHAP (SHapley Additive exPlanations) values provide a unified framework for interpreting model predictions. SHAP values reflect contributions to the model’s output (log-odds) for dropout. SHAP answers the questions: 

Which features contributed the most to predictions?

Why did the model give this specific prediction?

why SHAP?

-Exact for tree models (TreeSHAP)

-Works globally and locally

-Additive → contributions sum to the prediction

-Consistent feature attribution (theoretical guarantee)


In [ ]:
# SHAP Analysis
# Create SHAP explainer for XGBoost
explainer = shap.TreeExplainer(final_model)

# Calculate SHAP values for test set (sample if too large for memory)
# Using a sample for computational efficiency
sample_size = min(5000, len(X_test))
X_test_sample = X_test.sample(n=sample_size, random_state=42)
y_test_sample = y_test.loc[X_test_sample.index]

print(f"Calculating SHAP values for {sample_size} test samples...")
shap_values = explainer.shap_values(X_test_sample)
print("SHAP values calculated successfully!")

# Global feature importance based on mean absolute SHAP values
shap_importance = pd.DataFrame({
    'feature': X_test_sample.columns,
    'shap_importance': np.abs(shap_values).mean(axis=0)
}).sort_values('shap_importance', ascending=False)

print("Top 20 features by SHAP importance:")
print(shap_importance.head(20))

In [ ]:
# Standalone SHAP Summary Plot for detailed analysis
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_sample, max_display=20, show=False)
plt.title('SHAP Summary Plot - Feature Impact Distribution\n(Red = High feature values, Blue = Low feature values)', 
          fontsize=14, pad=20)
plt.tight_layout()
plt.show()

## 3.1 SHAP Results & Interpretation

SHAP values provide the most detailed and theoretically sound explanation of our model's predictions. Each SHAP value represents the contribution of a specific feature to moving the prediction away from the expected baseline.

Key insights from the SHAP analysis:

**Most Important Features (by mean absolute SHAP value):**
- Similar to previous methods, Block A academic performance features dominate
- The SHAP summary plot shows both the importance and the directional impact of features
- Positive SHAP values push predictions toward dropout, while negative SHAP values push toward retention; color indicates feature value (red = high, blue = low)

**Feature Impact Patterns:**
- **Total Credits Block A**: Lower credits strongly associated with higher dropout risk
- **Average Grade A**: Lower grades associated with higher dropout risk  
- **Course result flags**: Missing course results are significant risk factors

**SHAP Advantages:**
1. **Directional insights**: Unlike other methods, SHAP shows whether high/low feature values increase or decrease dropout risk
2. **Individual-level explanations**: SHAP enables individual-level explanations, although this analysis focuses on global patterns 
3. **Interaction effects**: While SHAP supports interaction analysis, this study focuses on main feature effects
4. **Theoretical soundness**: Based on cooperative game theory with desirable mathematical properties

## 4. Comparative Analysis

Let's compare the results from all three methods to identify the most robust dropout risk factors.

Note that in section 1 we included weight-based importance for diagnostic purposes to illustrate how frequently features are used during tree construction.
These values (i.e. weight and cover) are excluded from the aggregated importance score because they do not
measure predictive contribution and are not directly comparable to gain, permutation importance, or SHAP values.

In [ ]:
# Comparative analysis of all three methods
# Combine results from all three approaches
comparison_df = pd.DataFrame({
    'feature': feature_cols
})

# Add XGBoost importance (total gain, %)
comparison_df = comparison_df.merge(
    df_gain.rename(columns={'importance_total_gain_pct': 'xgb_total_gain_pct'})[
        ['feature', 'xgb_total_gain_pct']
    ],
    on='feature', how='left'
).fillna(0)

# Add XGBoost importance (weight, split frequency %; diagnostic only)
comparison_df = comparison_df.merge(
    df_weight.rename(columns={'importance_weight_pct': 'xgb_weight_pct'})[
        ['feature', 'xgb_weight_pct']
    ],
    on='feature', how='left'
).fillna(0)

# Add permutation importance
comparison_df = comparison_df.merge(
    perm_df.rename(columns={'importance_mean': 'perm_importance'})[
        ['feature', 'perm_importance']
    ],
    on='feature', how='left'
).fillna(0)

# Add SHAP importance
comparison_df = comparison_df.merge(
    shap_importance[['feature', 'shap_importance']],
    on='feature', how='left'
).fillna(0)

# Normalize importance scores to 0–1 scale for comparison
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

# Exclude weight from aggregation to ensure the average reflects
# predictive contribution rather than split frequency
importance_cols = ['xgb_total_gain_pct', 'perm_importance', 'shap_importance']
diagnostic_cols = ['xgb_weight_pct']

comparison_df[importance_cols + diagnostic_cols] = scaler.fit_transform(
    comparison_df[importance_cols + diagnostic_cols]
)

# Calculate average importance across predictive methods only
comparison_df['avg_importance'] = comparison_df[importance_cols].mean(axis=1)
comparison_df = comparison_df.sort_values('avg_importance', ascending=False)

# Display top 20 features across all methods
print("Top 20 Features - Comparative Analysis (Normalized 0–1 Scale):")
print(
    comparison_df.head(20)[
        ['feature'] + importance_cols + diagnostic_cols + ['avg_importance']
    ].round(3)
)


In [ ]:
# Visualize comparative analysis
plt.figure(figsize=(15, 10))

# Heatmap of top 20 features across all methods
top_20_comparison = comparison_df.head(20)

plt.subplot(2, 1, 1)

# // Include weight in the heatmap as a diagnostic row, even though it is excluded from the aggregated score
heatmap_cols = importance_cols + diagnostic_cols

heatmap_data = top_20_comparison[heatmap_cols].values.T
plt.imshow(heatmap_data, cmap='YlOrRd', aspect='auto')
plt.colorbar(label='Normalized Importance Score')

# // Update labels to match the heatmap column order, keeping weight explicitly marked as diagnostic
plt.yticks(
    range(len(heatmap_cols)),
    ['XGBoost (Total Gain %)', 'Permutation', 'SHAP', 'XGBoost (Weight %, Diagnostic)']
)

plt.xticks(
    range(len(top_20_comparison)),
    top_20_comparison['feature'], rotation=45, ha='right'
)
plt.title('Top 20 Features - Importance Heatmap Across Methods')

# Bar plot of average importance
plt.subplot(2, 1, 2)
plt.barh(range(len(top_20_comparison)), top_20_comparison['avg_importance'])
plt.yticks(range(len(top_20_comparison)), top_20_comparison['feature'])
plt.xlabel('Average Normalized Importance Score (Excludes Weight)')
# // Clarify that the aggregated score excludes weight so readers do not assume split frequency influences the average
plt.title('Top 20 Features - Average Importance Across Predictive Methods')
# // Align title with the aggregation definition (gain + permutation + SHAP only)
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

# Summary of most consistent important features
print("\nMost Consistent Important Features (appearing in top 10 of multiple methods):")

# // Keep weight in the consistency summary as a diagnostic comparator, but interpret it separately from predictive methods
top_10_per_method = {
    'XGBoost_TotalGain': set(df_gain.head(10)['feature']),
    'Permutation': set(perm_df.head(10)['feature']),
    'SHAP': set(shap_importance.head(10)['feature']),
    'XGBoost_Weight_Diagnostic': set(df_weight.head(10)['feature'])
}

# Find features appearing in top 10 of at least 3 methods
all_features = set()
for method_features in top_10_per_method.values():
    all_features.update(method_features)

consistent_features = []
for feature in all_features:
    count = sum(
        1 for method_features in top_10_per_method.values()
        if feature in method_features
    )
    if count >= 3:
        consistent_features.append((feature, count))

consistent_features.sort(key=lambda x: x[1], reverse=True)

for feature, count in consistent_features:
    print(f"- {feature}: appears in top 10 of {count}/4 methods")


## 5. Statistical Significance Testing

While the previous analyses identified important features for dropout prediction, we now assess their **statistical significance** through hypothesis testing. This complements the importance measures by determining which risk factors show statistically significant differences between dropout and non-dropout students.

We perform univariate tests on **original (unscaled) data** to ensure results are interpretable in meaningful units. For each top risk factor, we:

1. **Categorical features**: Chi-square test of independence + Cramér's V (effect size)
2. **Continuous features**: Mann-Whitney U test (non-parametric) + Cohen's d (effect size)

We use Mann-Whitney U instead of t-tests because student data often violates normality assumptions (e.g., grades skewed toward higher values, credits have ceiling effects).

**Be aware: the below code is still work in progress, and needs checking and interpretation!** \
**Be aware: below code needs UNSCALED data that does have the Dec 5th cut-off and 'set' marked present**

In [ ]:
# Load original (unscaled) data with target variable
# This data is before standardization to allow for interpretable statistical tests
df_original = pd.read_csv(PROCESSED_DIR / 'encoded_and_split_data.csv')

# Filter to test set only (same as used in importance analyses)
df_test_original = df_original[df_original['set'] == 'test'].copy()

# Verify we have the same samples
assert len(df_test_original) == len(test_df), "Test set size mismatch"

print(f"Loaded original test data: {len(df_test_original)} samples")
print(f"Dropout rate: {df_test_original[target_col].mean():.2%}")

# Get top 20 features from comparative analysis to focus statistical tests
top_features = comparison_df.head(20)['feature'].tolist()
print(f"\nWill perform statistical tests on top {len(top_features)} features")

In [ ]:
# Import statistical testing libraries
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu
import math

def cramers_v(chi2, n, r, c):
    """
    Calculate Cramér's V effect size for categorical associations.
    
    Parameters:
    chi2: Chi-square statistic
    n: Sample size
    r: Number of rows
    c: Number of columns
    
    Returns:
    Cramér's V (effect size between 0 and 1)
    """
    return math.sqrt(chi2 / (n * min(r - 1, c - 1)))

def cohens_d(group1, group2):
    """
    Calculate Cohen's d effect size for continuous variables.
    
    Parameters:
    group1, group2: Arrays of continuous values for two groups
    
    Returns:
    Cohen's d (standardized mean difference)
    """
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (np.mean(group1) - np.mean(group2)) / pooled_std

def interpret_effect_size(effect_size, test_type='cohens_d'):
    """
    Interpret effect size magnitude according to standard thresholds.
    
    Cohen's d: 0.2 (small), 0.5 (medium), 0.8 (large)
    Cramér's V: 0.1 (small), 0.3 (medium), 0.5 (large)
    """
    if test_type == 'cohens_d':
        abs_d = abs(effect_size)
        if abs_d < 0.2:
            return 'negligible'
        elif abs_d < 0.5:
            return 'small'
        elif abs_d < 0.8:
            return 'medium'
        else:
            return 'large'
    elif test_type == 'cramers_v':
        if effect_size < 0.1:
            return 'negligible'
        elif effect_size < 0.3:
            return 'small'
        elif effect_size < 0.5:
            return 'medium'
        else:
            return 'large'

print("Statistical testing functions defined successfully!")

In [ ]:
# Perform univariate statistical tests for each top feature
statistical_results = []

for feature in top_features:
    # Skip if feature not in original data
    if feature not in df_test_original.columns:
        print(f"Warning: {feature} not found in original data, skipping")
        continue
    
    # Split by dropout status
    dropout_group = df_test_original[df_test_original[target_col] == 1][feature]
    retained_group = df_test_original[df_test_original[target_col] == 0][feature]
    
    # Determine if feature is categorical or continuous
    # Consider feature categorical if it has fewer than 10 unique values
    n_unique = df_test_original[feature].nunique()
    is_categorical = n_unique < 10
    
    if is_categorical:
        # Chi-square test for categorical features
        contingency_table = pd.crosstab(
            df_test_original[feature],
            df_test_original[target_col]
        )
        
        # Skip if not enough variation (1 row or 1 column only)
        r, c = contingency_table.shape
        if r < 2 or c < 2:
            print(f"Warning: {feature} has insufficient variation for chi-square test, treating as continuous")
            is_categorical = False
        else:
            chi2, p_value, dof, expected = chi2_contingency(contingency_table)
            
            # Calculate Cramér's V effect size
            n = len(df_test_original)
            effect_size = cramers_v(chi2, n, r, c)
            effect_interpretation = interpret_effect_size(effect_size, 'cramers_v')
            
            statistical_results.append({
                'feature': feature,
                'test_type': 'Chi-Square',
                'statistic': chi2,
                'p_value': p_value,
                'effect_size': effect_size,
                'effect_size_type': "Cramér's V",
                'effect_interpretation': effect_interpretation,
                'n_unique_values': n_unique,
                'dropout_mean': dropout_group.mean(),
                'retained_mean': retained_group.mean()
            })
    
    if not is_categorical:
        # Mann-Whitney U test for continuous features
        dropout_clean = dropout_group.dropna()
        retained_clean = retained_group.dropna()
        
        # Skip if not enough data
        if len(dropout_clean) < 2 or len(retained_clean) < 2:
            print(f"Warning: {feature} has insufficient data, skipping")
            continue
            
        statistic, p_value = mannwhitneyu(
            dropout_clean,
            retained_clean,
            alternative='two-sided'
        )
        
        # Calculate Cohen's d effect size
        effect_size = cohens_d(dropout_clean, retained_clean)
        effect_interpretation = interpret_effect_size(effect_size, 'cohens_d')
        
        statistical_results.append({
            'feature': feature,
            'test_type': 'Mann-Whitney U',
            'statistic': statistic,
            'p_value': p_value,
            'effect_size': effect_size,
            'effect_size_type': "Cohen's d",
            'effect_interpretation': effect_interpretation,
            'n_unique_values': n_unique,
            'dropout_mean': dropout_group.mean(),
            'retained_mean': retained_group.mean()
        })

# Convert to DataFrame
stats_df = pd.DataFrame(statistical_results)

# Add significance indicator (Bonferroni-corrected alpha)
alpha = 0.05
bonferroni_alpha = alpha / len(top_features)
stats_df['significant'] = stats_df['p_value'] < bonferroni_alpha
stats_df['significant_uncorrected'] = stats_df['p_value'] < alpha

# Sort by p-value
stats_df = stats_df.sort_values('p_value')

print(f"\nCompleted statistical tests for {len(stats_df)} features")
print(f"Bonferroni-corrected alpha: {bonferroni_alpha:.4f}")
print(f"Significant features (Bonferroni): {stats_df['significant'].sum()}")
print(f"Significant features (uncorrected): {stats_df['significant_uncorrected'].sum()}")

In [ ]:
# Display statistical test results
print("\n" + "="*100)
print("STATISTICAL SIGNIFICANCE TESTS - TOP 20 RISK FACTORS")
print("="*100)

# Format for display
display_df = stats_df[[
    'feature', 'test_type', 'p_value', 'effect_size', 'effect_size_type',
    'effect_interpretation', 'dropout_mean', 'retained_mean', 'significant'
]].copy()

# Format p-values in scientific notation for very small values
display_df['p_value_formatted'] = display_df['p_value'].apply(
    lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}"
)

# Format means
display_df['dropout_mean'] = display_df['dropout_mean'].round(3)
display_df['retained_mean'] = display_df['retained_mean'].round(3)
display_df['effect_size'] = display_df['effect_size'].round(3)

# Rename for display
display_df = display_df.rename(columns={
    'dropout_mean': 'Mean (Dropout)',
    'retained_mean': 'Mean (Retained)',
    'effect_size': 'Effect Size',
    'effect_interpretation': 'Magnitude',
    'significant': 'Sig. (Bonf.)'
})

display(display_df[[
    'feature', 'test_type', 'p_value_formatted', 'Effect Size',
    'effect_size_type', 'Magnitude', 'Mean (Dropout)', 'Mean (Retained)', 'Sig. (Bonf.)'
]])

In [ ]:
# Visualize statistical significance and effect sizes
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. P-values (log scale)
ax1 = axes[0, 0]
colors = ['green' if sig else 'gray' for sig in stats_df['significant']]
ax1.barh(range(len(stats_df)), -np.log10(stats_df['p_value']), color=colors)
ax1.axvline(-np.log10(bonferroni_alpha), color='red', linestyle='--', 
            linewidth=2, label=f'Bonferroni threshold (α={bonferroni_alpha:.4f})')
ax1.axvline(-np.log10(alpha), color='orange', linestyle='--', 
            linewidth=1, label=f'Uncorrected α={alpha}')
ax1.set_yticks(range(len(stats_df)))
ax1.set_yticklabels(stats_df['feature'], fontsize=9)
ax1.set_xlabel('-log10(p-value)', fontsize=11)
ax1.set_title('Statistical Significance of Risk Factors\n(Higher = More Significant)', fontsize=12, fontweight='bold')
ax1.legend()
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3)

# 2. Effect sizes
ax2 = axes[0, 1]
# Color by effect magnitude
effect_colors = stats_df['effect_interpretation'].map({
    'negligible': 'lightgray',
    'small': 'lightblue',
    'medium': 'orange',
    'large': 'red'
})
ax2.barh(range(len(stats_df)), stats_df['effect_size'].abs(), color=effect_colors)
ax2.set_yticks(range(len(stats_df)))
ax2.set_yticklabels(stats_df['feature'], fontsize=9)
ax2.set_xlabel('Effect Size (Absolute)', fontsize=11)
ax2.set_title('Effect Size Magnitude\n(Gray=Negligible, Blue=Small, Orange=Medium, Red=Large)', 
              fontsize=12, fontweight='bold')
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)

# 3. Mean differences (continuous features only)
ax3 = axes[1, 0]
continuous_features = stats_df[stats_df['test_type'] == 'Mann-Whitney U'].copy()
if len(continuous_features) > 0:
    mean_diff = continuous_features['dropout_mean'] - continuous_features['retained_mean']
    colors_diff = ['red' if x > 0 else 'blue' for x in mean_diff]
    ax3.barh(range(len(continuous_features)), mean_diff, color=colors_diff, alpha=0.7)
    ax3.set_yticks(range(len(continuous_features)))
    ax3.set_yticklabels(continuous_features['feature'], fontsize=9)
    ax3.set_xlabel('Mean Difference (Dropout - Retained)', fontsize=11)
    ax3.set_title('Mean Differences for Continuous Features\n(Red=Higher in Dropout, Blue=Higher in Retained)', 
                  fontsize=12, fontweight='bold')
    ax3.axvline(0, color='black', linestyle='-', linewidth=0.8)
    ax3.invert_yaxis()
    ax3.grid(axis='x', alpha=0.3)
else:
    ax3.text(0.5, 0.5, 'No continuous features in top 20', 
             ha='center', va='center', transform=ax3.transAxes)
    ax3.axis('off')

# 4. Combined: Effect size vs -log10(p-value) scatter
ax4 = axes[1, 1]
scatter_colors = stats_df['significant'].map({True: 'green', False: 'gray'})
ax4.scatter(stats_df['effect_size'].abs(), -np.log10(stats_df['p_value']), 
           c=scatter_colors, s=100, alpha=0.6, edgecolors='black')
ax4.axhline(-np.log10(bonferroni_alpha), color='red', linestyle='--', 
            linewidth=1, label='Bonferroni threshold')
ax4.set_xlabel('Effect Size (Absolute)', fontsize=11)
ax4.set_ylabel('-log10(p-value)', fontsize=11)
ax4.set_title('Statistical Significance vs Effect Size\n(Green=Significant after Bonferroni correction)', 
              fontsize=12, fontweight='bold')
ax4.legend()
ax4.grid(alpha=0.3)

# Annotate top features
for idx, row in stats_df.head(5).iterrows():
    ax4.annotate(row['feature'], 
                xy=(abs(row['effect_size']), -np.log10(row['p_value'])),
                xytext=(5, 5), textcoords='offset points', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.3))

plt.tight_layout()
plt.show()

### 5.1 Statistical Significance Results & Interpretation

**Key Findings:**

1. **Multiple Testing Correction**: We applied Bonferroni correction to control for false positives when testing multiple hypotheses. With 20 tests, the corrected significance threshold is α = 0.05/20 = 0.0025.

2. **P-value Interpretation**: 
   - Values shown as scientific notation (e.g., 1.23e-10) indicate extremely strong evidence against the null hypothesis
   - Green bars in the visualization indicate features that remain significant after Bonferroni correction
   - The -log10 transformation means higher bars = stronger statistical significance

3. **Effect Size Interpretation**:
   - **Cohen's d** (continuous): Standardized mean difference between dropout and retained groups
     - Small: 0.2-0.5 | Medium: 0.5-0.8 | Large: >0.8
   - **Cramér's V** (categorical): Association strength between feature and dropout status
     - Small: 0.1-0.3 | Medium: 0.3-0.5 | Large: >0.5

4. **Statistical vs Practical Significance**:
   - Statistical significance (low p-value) ≠ practical importance
   - Effect size quantifies the **magnitude** of differences, which is often more meaningful for intervention planning
   - Features can be statistically significant but have small practical effects, or vice versa

5. **Mean Differences** (for continuous features):
   - Positive values: Dropout students have higher average values than retained students
   - Negative values: Retained students have higher average values than dropout students
   - Magnitude indicates the real-world difference in original units (e.g., credits, grade points)

**Next Steps**: The scatter plot (bottom-right) helps identify features that are both statistically significant AND have large practical effects - these are the most actionable risk factors for intervention.

## 6. Protective Factors Analysis

While the previous section focused on **risk factors** (features associated with higher dropout probability), we now examine **protective factors** - features that are associated with student retention and lower dropout risk.

We identify protective factors by analyzing the **directional impact** of features on dropout predictions using SHAP values:
- Protective factors have **negative mean SHAP contributions** (push predictions toward retention)
- These are features where higher values (or specific categories) are associated with staying enrolled

This analysis complements the risk factor analysis and may reveal intervention opportunities focused on strengthening protective factors rather than just mitigating risks.

In [ ]:
# Identify protective factors using mean SHAP values
# Protective factors have negative mean SHAP (push toward retention, class 0)
mean_shap_per_feature = pd.DataFrame({
    'feature': X_test_sample.columns,
    'mean_shap': shap_values.mean(axis=0)  # Mean SHAP value across all samples
})

# Protective factors: negative mean SHAP (reduce dropout probability)
protective_shap = mean_shap_per_feature[
    mean_shap_per_feature['mean_shap'] < 0
].sort_values('mean_shap')  # Most negative = strongest protective effect

print(f"Total features analyzed: {len(mean_shap_per_feature)}")
print(f"Protective factors (negative mean SHAP): {len(protective_shap)}")
print(f"Risk factors (positive mean SHAP): {(mean_shap_per_feature['mean_shap'] > 0).sum()}")
print(f"\nTop 20 Protective Factors by Mean SHAP Value:")
print("="*70)

top_20_protective = protective_shap.head(20)
display(top_20_protective)

In [ ]:
# Perform statistical significance testing on protective factors
protective_features = top_20_protective['feature'].tolist()
protective_results = []

for feature in protective_features:
    # Skip if feature not in original data
    if feature not in df_test_original.columns:
        print(f"Warning: {feature} not found in original data, skipping")
        continue
    
    # Split by dropout status
    dropout_group = df_test_original[df_test_original[target_col] == 1][feature]
    retained_group = df_test_original[df_test_original[target_col] == 0][feature]
    
    # Determine if feature is categorical or continuous
    n_unique = df_test_original[feature].nunique()
    is_categorical = n_unique < 10
    
    if is_categorical:
        # Chi-square test for categorical features
        contingency_table = pd.crosstab(
            df_test_original[feature],
            df_test_original[target_col]
        )
        
        # Skip if not enough variation
        r, c = contingency_table.shape
        if r < 2 or c < 2:
            print(f"Warning: {feature} has insufficient variation for chi-square test, treating as continuous")
            is_categorical = False
        else:
            chi2, p_value, dof, expected = chi2_contingency(contingency_table)
            
            # Calculate Cramér's V effect size
            n = len(df_test_original)
            effect_size = cramers_v(chi2, n, r, c)
            effect_interpretation = interpret_effect_size(effect_size, 'cramers_v')
            
            protective_results.append({
                'feature': feature,
                'test_type': 'Chi-Square',
                'statistic': chi2,
                'p_value': p_value,
                'effect_size': effect_size,
                'effect_size_type': "Cramér's V",
                'effect_interpretation': effect_interpretation,
                'n_unique_values': n_unique,
                'dropout_mean': dropout_group.mean(),
                'retained_mean': retained_group.mean(),
                'mean_shap': top_20_protective[top_20_protective['feature'] == feature]['mean_shap'].values[0]
            })
    
    if not is_categorical:
        # Mann-Whitney U test for continuous features
        dropout_clean = dropout_group.dropna()
        retained_clean = retained_group.dropna()
        
        # Skip if not enough data
        if len(dropout_clean) < 2 or len(retained_clean) < 2:
            print(f"Warning: {feature} has insufficient data, skipping")
            continue
            
        statistic, p_value = mannwhitneyu(
            dropout_clean,
            retained_clean,
            alternative='two-sided'
        )
        
        # Calculate Cohen's d effect size
        effect_size = cohens_d(dropout_clean, retained_clean)
        effect_interpretation = interpret_effect_size(effect_size, 'cohens_d')
        
        protective_results.append({
            'feature': feature,
            'test_type': 'Mann-Whitney U',
            'statistic': statistic,
            'p_value': p_value,
            'effect_size': effect_size,
            'effect_size_type': "Cohen's d",
            'effect_interpretation': effect_interpretation,
            'n_unique_values': n_unique,
            'dropout_mean': dropout_group.mean(),
            'retained_mean': retained_group.mean(),
            'mean_shap': top_20_protective[top_20_protective['feature'] == feature]['mean_shap'].values[0]
        })

# Convert to DataFrame
protective_stats_df = pd.DataFrame(protective_results)

# Add significance indicator (Bonferroni-corrected alpha)
alpha = 0.05
bonferroni_alpha_protective = alpha / len(protective_features)
protective_stats_df['significant'] = protective_stats_df['p_value'] < bonferroni_alpha_protective
protective_stats_df['significant_uncorrected'] = protective_stats_df['p_value'] < alpha

# Sort by p-value
protective_stats_df = protective_stats_df.sort_values('p_value')

print(f"\nCompleted statistical tests for {len(protective_stats_df)} protective factors")
print(f"Bonferroni-corrected alpha: {bonferroni_alpha_protective:.4f}")
print(f"Significant features (Bonferroni): {protective_stats_df['significant'].sum()}")
print(f"Significant features (uncorrected): {protective_stats_df['significant_uncorrected'].sum()}")

In [ ]:
# Display protective factors statistical test results
print("\n" + "="*100)
print("STATISTICAL SIGNIFICANCE TESTS - TOP 20 PROTECTIVE FACTORS")
print("="*100)

# Format for display
display_df_protective = protective_stats_df[[
    'feature', 'test_type', 'p_value', 'effect_size', 'effect_size_type',
    'effect_interpretation', 'dropout_mean', 'retained_mean', 'mean_shap', 'significant'
]].copy()

# Format p-values
display_df_protective['p_value_formatted'] = display_df_protective['p_value'].apply(
    lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}"
)

# Format numeric columns
display_df_protective['dropout_mean'] = display_df_protective['dropout_mean'].round(3)
display_df_protective['retained_mean'] = display_df_protective['retained_mean'].round(3)
display_df_protective['effect_size'] = display_df_protective['effect_size'].round(3)
display_df_protective['mean_shap'] = display_df_protective['mean_shap'].round(4)

# Rename for display
display_df_protective = display_df_protective.rename(columns={
    'dropout_mean': 'Mean (Dropout)',
    'retained_mean': 'Mean (Retained)',
    'effect_size': 'Effect Size',
    'effect_interpretation': 'Magnitude',
    'mean_shap': 'Mean SHAP',
    'significant': 'Sig. (Bonf.)'
})

display(display_df_protective[[
    'feature', 'test_type', 'p_value_formatted', 'Effect Size',
    'effect_size_type', 'Magnitude', 'Mean (Dropout)', 'Mean (Retained)', 
    'Mean SHAP', 'Sig. (Bonf.)'
]])

In [ ]:
# Visualize protective factors: statistical significance and effect sizes
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. P-values (log scale) for protective factors
ax1 = axes[0, 0]
colors = ['darkgreen' if sig else 'gray' for sig in protective_stats_df['significant']]
ax1.barh(range(len(protective_stats_df)), -np.log10(protective_stats_df['p_value']), color=colors)
ax1.axvline(-np.log10(bonferroni_alpha_protective), color='red', linestyle='--', 
            linewidth=2, label=f'Bonferroni threshold (α={bonferroni_alpha_protective:.4f})')
ax1.axvline(-np.log10(alpha), color='orange', linestyle='--', 
            linewidth=1, label=f'Uncorrected α={alpha}')
ax1.set_yticks(range(len(protective_stats_df)))
ax1.set_yticklabels(protective_stats_df['feature'], fontsize=9)
ax1.set_xlabel('-log10(p-value)', fontsize=11)
ax1.set_title('Statistical Significance of Protective Factors\n(Higher = More Significant)', 
              fontsize=12, fontweight='bold')
ax1.legend()
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3)

# 2. Effect sizes
ax2 = axes[0, 1]
effect_colors = protective_stats_df['effect_interpretation'].map({
    'negligible': 'lightgray',
    'small': 'lightblue',
    'medium': 'orange',
    'large': 'red'
})
ax2.barh(range(len(protective_stats_df)), protective_stats_df['effect_size'].abs(), color=effect_colors)
ax2.set_yticks(range(len(protective_stats_df)))
ax2.set_yticklabels(protective_stats_df['feature'], fontsize=9)
ax2.set_xlabel('Effect Size (Absolute)', fontsize=11)
ax2.set_title('Effect Size Magnitude\n(Gray=Negligible, Blue=Small, Orange=Medium, Red=Large)', 
              fontsize=12, fontweight='bold')
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)

# 3. Mean SHAP values (protective effect strength)
ax3 = axes[1, 0]
shap_colors = ['darkgreen' if x < -0.01 else 'lightgreen' for x in protective_stats_df['mean_shap']]
ax3.barh(range(len(protective_stats_df)), protective_stats_df['mean_shap'], color=shap_colors)
ax3.set_yticks(range(len(protective_stats_df)))
ax3.set_yticklabels(protective_stats_df['feature'], fontsize=9)
ax3.set_xlabel('Mean SHAP Value (Negative = Protective)', fontsize=11)
ax3.set_title('Protective Effect Strength\n(More Negative = Stronger Protection Against Dropout)', 
              fontsize=12, fontweight='bold')
ax3.axvline(0, color='black', linestyle='-', linewidth=0.8)
ax3.invert_yaxis()
ax3.grid(axis='x', alpha=0.3)

# 4. Combined: Effect size vs -log10(p-value) scatter
ax4 = axes[1, 1]
scatter_colors = protective_stats_df['significant'].map({True: 'darkgreen', False: 'gray'})
ax4.scatter(protective_stats_df['effect_size'].abs(), 
           -np.log10(protective_stats_df['p_value']), 
           c=scatter_colors, s=100, alpha=0.6, edgecolors='black')
ax4.axhline(-np.log10(bonferroni_alpha_protective), color='red', linestyle='--', 
            linewidth=1, label='Bonferroni threshold')
ax4.set_xlabel('Effect Size (Absolute)', fontsize=11)
ax4.set_ylabel('-log10(p-value)', fontsize=11)
ax4.set_title('Statistical Significance vs Effect Size\n(Green=Significant after Bonferroni correction)', 
              fontsize=12, fontweight='bold')
ax4.legend()
ax4.grid(alpha=0.3)

# Annotate top features
for idx, row in protective_stats_df.head(5).iterrows():
    ax4.annotate(row['feature'][:30], 
                xy=(abs(row['effect_size']), -np.log10(row['p_value'])),
                xytext=(5, 5), textcoords='offset points', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.5))

plt.tight_layout()
plt.show()

### 6.1 Protective Factors Results & Interpretation

**Key Findings:**

1. **Identification Method**: Protective factors were identified using **mean SHAP values < 0**, indicating features that on average push predictions away from dropout toward retention.

2. **Statistical Validation**: We applied the same rigorous testing as for risk factors:
   - Bonferroni-corrected significance threshold to control false positives
   - Effect size calculations to assess practical importance
   - Separate tests for categorical (Chi-square + Cramér's V) and continuous (Mann-Whitney U + Cohen's d) features

3. **Interpretation Guidance**:
   - **Negative Mean SHAP**: The more negative, the stronger the protective effect
   - **Mean Differences**: For protective factors, we typically expect:
     - Retained students to have **higher** values than dropout students (negative Cohen's d)
     - OR specific categorical values more prevalent in retained students
   
4. **Protection vs Risk**: 
   - Protective factors complement risk factors - they represent the "other side of the coin"
   - Some features may appear in both analyses if they have bidirectional effects (e.g., low grades = risk, high grades = protective)
   - The mean SHAP value indicates the **average direction** of the feature's contribution

5. **Intervention Implications**:
   - Risk factor interventions: Reduce or mitigate negative influences
   - Protective factor interventions: Strengthen or amplify positive influences
   - Both approaches are valuable for comprehensive dropout prevention strategies

In [ ]:
# Summary: Protective factors with both high statistical significance AND large effect sizes
print("="*100)
print("ACTIONABLE PROTECTIVE FACTORS: Statistically Significant with Large/Medium Effect Sizes")
print("="*100)
print("\nThese features are the most promising targets for enhancement/strengthening:")
print("(Both statistically significant after Bonferroni correction AND have practical importance)\n")

# Filter for significant protective factors with at least medium effect size
actionable_protective = protective_stats_df[
    (protective_stats_df['significant'] == True) & 
    (protective_stats_df['effect_interpretation'].isin(['medium', 'large']))
].copy()

if len(actionable_protective) > 0:
    for idx, row in actionable_protective.iterrows():
        print(f"\n{row['feature']}")
        print(f"  • Test: {row['test_type']}")
        print(f"  • p-value: {row['p_value']:.2e}")
        print(f"  • Effect size: {row['effect_size']:.3f} ({row['effect_size_type']}) - {row['effect_interpretation'].upper()}")
        print(f"  • Mean SHAP: {row['mean_shap']:.4f} (negative = protective)")
        print(f"  • Mean dropout group: {row['dropout_mean']:.3f}")
        print(f"  • Mean retained group: {row['retained_mean']:.3f}")
        
        # Interpretation
        if row['test_type'] == 'Mann-Whitney U':
            diff = row['retained_mean'] - row['dropout_mean']
            direction = "higher" if diff > 0 else "lower"
            print(f"  → Retained students have {direction} values (diff: {diff:.3f})")
else:
    print("No protective factors meet both criteria (Bonferroni-significant + medium/large effect)")

print("\n" + "="*100)
print(f"\nTotal protective factors tested: {len(protective_stats_df)}")
print(f"Statistically significant (Bonferroni-corrected): {protective_stats_df['significant'].sum()}")
print(f"With medium/large effect sizes: {protective_stats_df['effect_interpretation'].isin(['medium', 'large']).sum()}")
print(f"Meeting both criteria: {len(actionable_protective)}")